# Baseline diagnostics
Run 01, then 02 or 03. Each notebook uses a fresh kernel. Smoke mode and validation-only evaluation are defaults. Outputs are local and ignored. Final test evaluation is explicitly opt-in; external evaluation is deferred.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

project_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/datasets.py').exists())
sys.path.insert(0, str(project_dir))
from src.datasets import dataset_path
from src.audio.features import SETTINGS, mfcc_features
from src.audio.workflow import prepare_frames, fit_export

FULL_RUN = False
FINAL_EVALUATION = False  # Opt in only after model choices are frozen.
output_dir = project_dir / 'outputs/audio' / ('full' if FULL_RUN else 'smoke') / ('final' if FINAL_EVALUATION else 'development')
la_dir = dataset_path('asvspoof19_la')
frames = prepare_frames(la_dir, output_dir, full=FULL_RUN, final=FINAL_EVALUATION)
display(pd.DataFrame({s: f['label'].value_counts() for s, f in frames.items()}))


In [ ]:
import librosa
import soundfile as sf
from src.audio.evaluate import new_classifier, evaluate_scores
import json

baseline_settings = json.loads((output_dir / 'mfcc_settings.json').read_text())
threshold = baseline_settings['threshold']
import joblib
model = joblib.load(output_dir / 'mfcc_classifier.joblib')
validation = frames['validation']
audit = validation[['sample_id', 'label']].copy()
audit['duration'] = [sf.info(path).duration for path in validation['path']]
audit['silence_fraction'] = [1 - sum(end-start for start, end in librosa.effects.split(librosa.load(path, sr=16000)[0], top_db=30)) / sf.info(path).frames for path in validation['path']]
display(audit.groupby('label')[['duration', 'silence_fraction']].mean())
results = []
for condition in ['clean', 'noise', 'mulaw']:
    values = mfcc_features(validation['path'], condition=condition, seed=42)
    scores = model.predict_proba(values)[:, 1]
    results.append({'experiment': condition, **evaluate_scores(validation['label'], scores, threshold)})
    if condition == 'clean':
        audit['score'] = scores
for trim in [True, False]:
    train_X = mfcc_features(frames['train']['path'], trim=trim, seed=42)
    valid_X = mfcc_features(validation['path'], trim=trim, seed=42)
    comparison = new_classifier(42).fit(train_X, frames['train']['label'])
    scores = comparison.predict_proba(valid_X)[:, 1]
    results.append({'experiment': f'trim={trim}', **evaluate_scores(validation['label'], scores, threshold)})
clean_X = mfcc_features(frames['train']['path'], seed=42)
noisy_X = mfcc_features(frames['train']['path'], condition='noise', seed=42)
augmented = new_classifier(42).fit(np.vstack([clean_X, noisy_X]), np.tile(frames['train']['label'], 2))
scores = augmented.predict_proba(mfcc_features(validation['path'], seed=42))[:, 1]
results.append({'experiment': 'noise_augmentation', **evaluate_scores(validation['label'], scores, threshold)})
display(pd.DataFrame(results))
audit['error'] = (audit['score'] >= threshold).astype(int) != audit['label']
display(audit.loc[audit['error']].sort_values('score'))
